### 🏷️ **Credits & License**

- 🔗 OmniVoice GitHub Repository
- 🤗 lllllce on Hugging Face
- 📄 License: Provided under the Apache License 2.0

### ⚠️ **Usage Disclaimer**
Use of this voice cloning model is subject to strict ethical and legal standards.

By using this tool, you agree **not to** engage in:
- Fraud or deception
- Non-consensual impersonation
- Illegal activities
- Harmful or misleading content


In [ ]:
#@title Install OmniVoice + API Dependencies
%cd /content/

!rm -rf omnivoice-colab
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git

%cd /content/omnivoice-colab

# Clone OmniVoice repo
!rm -rf OmniVoice
!git clone https://github.com/NeuralFalconYT/OmniVoice.git

# Install ffmpeg for mp3/m4a/ogg support
!apt-get update -qq
!apt-get install -y ffmpeg

# Install requirements
!pip install -q -r colab.txt

# API packages
!pip install -q fastapi uvicorn[standard] python-multipart httpx requests aiofiles nest-asyncio python-dotenv pydub soundfile librosa

# Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# GPU Check
import torch

print('========== SYSTEM INFO ==========')

if torch.cuda.is_available():
    print('✅ GPU ENABLED')
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('❌ GPU NOT ENABLED!\nRuntime → Change runtime type → GPU')

# Fix OmniVoice import
import sys
sys.path.append('/content/omnivoice-colab/OmniVoice')

from IPython.display import clear_output
clear_output()

print('✅ Installation complete!')


In [ ]:
#@title Run OmniVoice API Server + Public URL
%cd /content/omnivoice-colab

# Download latest app.py
!wget -q -O app.py https://raw.githubusercontent.com/hahyt6644-gif/omnivoice-colab/main/app.py

# Kill previous servers
!pkill -f 'python app.py' || true
!pkill -f 'uvicorn' || true
!pkill -f 'cloudflared' || true

import subprocess
import threading
import time
import requests

print('🚀 Starting OmniVoice Server...')

# Start API server
server = subprocess.Popen(['python', 'app.py'])

# Wait for server
time.sleep(15)

# Start Cloudflare tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('\n🌍 Waiting for public URL...\n')

url = None
for _ in range(60):
    try:
        r = requests.get('http://localhost:4040/api/tunnels', timeout=2)
        tunnels = r.json()
        if tunnels.get('tunnels'):
            url = tunnels['tunnels'][0]['public_url']
            break
    except:
        pass

    line = tunnel.stdout.readline()
    if 'trycloudflare.com' in line:
        import re
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break

if url:
    print('=' * 60)
    print('✅ PUBLIC URL READY')
    print(url)
    print('\nClone API:')
    print(url + '/api/clone')
    print('\nDesign API:')
    print(url + '/api/design')
    print('\nUI:')
    print(url + '/ui')
    print('=' * 60)
else:
    print('❌ Failed to get public URL')
